### Latin square Simulation

- Latin square is the square with all rows and columns has distinct symbol: https://en.wikipedia.org/wiki/Latin_square
- In this notebook, the author will simulate the latent square for reasonable dimensions <= 11 and provide an est for total number of latent squares using SMC
- Additionally, the same idea will be applied to solve a sudoku puzzle. 

Let's start with some setup! Define:  
- $\pi_0$ is the uniform distribution of all squares n x n with each row is already a permutation from $1 - n$
so that we only need to consider the validity of columns.

- $\pi_1$ is the uniform distribution of all Latin squares n x n.

We will start from $\pi_0$ to generate samples from $\pi_T$

- Let's say if we directly apply the ideas of SMCS by building a list of intermediate distributions $\pi_0^{(1 - \lambda)} * \pi_1^{\lambda}$, what happen is that for all lambdas > 0, all intermediate dists will be collapsed to the target distribution as its pdf is exactly the same aft normalizing (i.e. = 0 if the square is non-Latin and = 1 otherwise)

The idea is interesting as we will try to not directly sample from our target dist but from an "approximate" version of it. 
Let's consider: 

$target = e^{-lambda * score(X)}$ with lambda is big enough value let say 1000. 

- Score(X) here is a score function to measure how "bad" a sample from a Latin properties.
- When X is Latin square, score(X) will be zero and target = 1, otherwise, this value is very small, thanks to the big lambda. By doing SMC on this alternative dist, most of the samples we received will expect to be Latin.

In [1]:
from libs import MCMC, SMC, AcceptanceTracker

First of all, define a scorer class to maintain all scorer methods we will have 

In [2]:
from jax import numpy as jnp

import numpy as np 

class Scorer:
    @staticmethod
    def score_repetitive_columns(flat_board: jnp.array):
        n = np.sqrt(len(flat_board)).astype(int)
        assert len(flat_board) == n * n
        
        score = 0 
        for c in range(n):
            # symbols are 1..n, so the mask needs n + 1 slots
            seen = jnp.zeros(n + 1, dtype=bool)
            for x in flat_board[c::n]:
                score += jnp.where(seen[x], 1, 0)
                seen = seen.at[x].set(True)
        return score 

    @staticmethod
    def score_repetitive_rows(flat_board: jnp.array):
        n = np.sqrt(len(flat_board)).astype(int)
        assert len(flat_board) == n * n
        
        score = 0 
        for r in range(n):
            # symbols are 1..n, so the mask needs n + 1 slots
            seen = jnp.zeros(n + 1, dtype=bool)
            for x in flat_board[0::n]:
                score += jnp.where(seen[x], 1, 0)
                seen = seen.at[x].set(True)
        return score 

    @staticmethod
    def score_sudoku(flat_board: jnp.array):
        assert len(flat_board) == 9 * 9

        score = Scorer.score_repetitive_columns(flat_board) 

        # conflict for each 3 x 3 square
        for r in range(0, 9, 3):
            for c in range(0, 9, 3): 
                seen = jnp.zeros(10, dtype = bool)
                for dr in range(0, 3):
                    for dc in range(0, 3):
                        pos = (r + dr) * 9 + (c + dc)
                        score += jnp.where(seen[flat_board[pos]], 1, 0)
                        seen = seen.at[flat_board[pos]].set(True)
                
        return score  

assert Scorer.score_repetitive_columns(jnp.array([1,2,1,2])) == 2
assert Scorer.score_repetitive_columns(jnp.array([1,2,2,1])) == 0

Declare an abstract class for any solver

In [3]:
from abc import ABC, abstractmethod
import jax

class ISolver(ABC):
    def __init__(self, n: int, target_dist_logpdf, prior_dist_logpdf, proposed_fn):
        self.__n = n
        self.__key = random.key(100)
        self.smc = SMC(
            dims = n * n, 
            target_dist_logpdf = target_dist_logpdf, 
            prior_dist_logpdf = prior_dist_logpdf, 
            proposed_fn = proposed_fn, 
            key = self._split_key()[0]
        )
        self.log_z = 0.0

    def _split_key(self, n = 2):
        split_keys = random.split(self.__key, n)
        self.__key = split_keys[0]
        return split_keys[1:]

    @abstractmethod
    def _generate_initial_state(self, key):
        pass

    def _generate_initial_states(self, no_samples: int):
        sub_keys = self._split_key(no_samples + 1)
        states = jax.vmap(
            lambda key: self._generate_initial_state(key)
        )(sub_keys)
        return states

    def run_smc(self, no_samples: int):
        samples = self._generate_initial_states(no_samples)
        assert samples.shape == (no_samples, self.__n * self.__n)
        
        self.smc.reset(samples = samples)
        lam_list, diff_log_z = self.smc.build_intermediate_dists(max_steps=64, n_bisect=30, mcmc_iters=50)
        self.log_z += diff_log_z
        return lam_list

    def get_current_sample_list(self):
        return self.smc.get_current_sample_list()

    def est_log_total(self):
        return self.log_z

#### Latin Square Solver

Secondly, define the prior/target dist logpdfs we will sample from as well as the proposed_fn to move to the next state


In [4]:
import jax
import jax.numpy as jnp
from jax import random

import numpy as np 

def prior_dist_logpdf(flat_board: jnp.array):
    """
    Supports:
        flat_board.shape == (N,)
        flat_board.shape == (B, N)
    """
    board_size = flat_board.shape[-1]
    n = int(np.sqrt(board_size))

    log_prob = -n * jnp.log(jnp.arange(1, n + 1)).sum()

    if flat_board.ndim == 1:
        return log_prob
    else:
        return jnp.full((flat_board.shape[0],), log_prob)

def target_dist_logpdf(flat_board: jnp.array, score_fn, lam=1000):
    """
    Supports:
        (N,)  -> scalar
        (B,N) -> (B,)
    """
    if flat_board.ndim == 1:
        return -lam * score_fn(flat_board)
    else:
        scores = jax.vmap(score_fn)(flat_board)
        return -lam * scores

In [5]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np

def proposed_fn(flat_board: jnp.array, key):
    """
    Supports:
        (N,)
        (B,N)
    """

    def propose_one(board: np.array, key):
        board_size = board.shape[0]
        n = jnp.sqrt(board_size).astype(int)

        key1, key2 = random.split(key)

        row = random.randint(key1, (), 0, n)
        col = random.randint(key2, (), 0, n - 1)

        i = row * n + col
        j = i + 1

        board_i, board_j = board[i], board[j]
        board = board.at[i].set(board_j)
        board = board.at[j].set(board_i)
        return board 

    if flat_board.ndim == 1:
        return propose_one(flat_board, key)

    else:
        keys = random.split(key, flat_board.shape[0])
        return jax.vmap(propose_one)(flat_board, keys)

In [6]:
from jax import random, vmap
from jax import numpy as jnp 
from functools import partial

class LatinSquareSampler(ISolver): 
    def __init__(self, n: int, score_fn):
        self.__n = n 
        super().__init__(
            n = n, 
            target_dist_logpdf = partial(target_dist_logpdf, score_fn = score_fn), 
            prior_dist_logpdf = prior_dist_logpdf, 
            proposed_fn = proposed_fn
        )

    def _generate_initial_state(self, key):
        sub_keys = random.split(key, num = self.__n)
        perm = vmap(
            lambda key: random.permutation(key, jnp.arange(1, self.__n + 1)) 
        )(sub_keys)
        return perm.reshape(-1)  

Now let's test our solver!

In [7]:
from jax import numpy as jnp

grouth_truth = [
    1, 
    2, 
    12, 
    576, 
    161280, 
    812851200, 
    61479419904000, 
    108776032459082956800, 
    5524751496156892842531225600, 
    9982437658213039871725064756920320000, 
    776966836171770144107444346734230682311065600000
]

log_grouth_truth = [
    0.0,
    0.6931471805599453,
    2.4849066497880004,
    6.356107660695891,
    11.989476363991853,
    20.51524607852745,
    31.74754634896515,
    46.13891007716285,
    63.87674183985093,
    85.1927379585188,
    59.61662636289668,
]

In [8]:
NDIMS = 9
latin_solver = LatinSquareSampler(
    n = NDIMS, 
    score_fn = Scorer.score_repetitive_columns
)

print(f"before log_z: {latin_solver.log_z}")

lam_list = latin_solver.run_smc(
    no_samples = 300_000
)

print(f"aft log_z: {latin_solver.log_z}")

samples = latin_solver.get_current_sample_list()

before log_z: 0.0
Current loop value: 0.0
Current loop value: 0.0001816609874367714
Current loop value: 0.0003644566750153899
Current loop value: 0.0005487597081810236
Current loop value: 0.0007348456420004368
Current loop value: 0.0009235994657501578
Current loop value: 0.0011150187347084284
Current loop value: 0.0013091540895402431
Current loop value: 0.0015058207791298628
Current loop value: 0.0017047177534550428
Current loop value: 0.0019065055530518293
Current loop value: 0.0021098458673805
Current loop value: 0.002315057907253504
Current loop value: 0.0025211693719029427
Current loop value: 0.00272967666387558
Current loop value: 0.002943237777799368
Current loop value: 0.003167572431266308
Current loop value: 0.00341263459995389
Current loop value: 0.003699858672916889
Current loop value: 0.004092961549758911
aft log_z: 63.663944244384766


Display some Latin square samples 

In [9]:
for sample in samples[:10]:
    print(sample.reshape(NDIMS, NDIMS), "\n")

[[8 3 9 5 1 2 4 7 6]
 [7 9 1 3 5 6 8 4 2]
 [1 8 7 9 3 4 6 2 5]
 [4 5 3 6 9 8 2 1 7]
 [6 7 2 1 4 9 5 3 8]
 [9 6 4 2 7 5 1 8 3]
 [2 1 8 4 6 3 7 5 9]
 [5 4 6 8 2 7 3 9 1]
 [3 2 5 7 8 1 9 6 4]] 

[[8 3 9 5 1 2 4 7 6]
 [7 9 1 3 5 6 8 4 2]
 [1 8 7 9 3 4 6 2 5]
 [4 5 3 6 9 8 2 1 7]
 [6 7 2 1 4 9 5 3 8]
 [9 6 4 2 7 5 1 8 3]
 [2 1 8 4 6 3 7 5 9]
 [5 4 6 8 2 7 3 9 1]
 [3 2 5 7 8 1 9 6 4]] 

[[8 3 9 5 1 2 4 7 6]
 [7 9 1 3 5 6 8 4 2]
 [1 8 7 9 3 4 6 2 5]
 [4 5 3 6 9 8 2 1 7]
 [6 7 2 1 4 9 5 3 8]
 [9 6 4 2 7 5 1 8 3]
 [2 1 8 4 6 3 7 5 9]
 [5 4 6 8 2 7 3 9 1]
 [3 2 5 7 8 1 9 6 4]] 

[[4 9 5 2 6 7 3 8 1]
 [5 1 6 3 7 8 9 4 2]
 [9 8 3 6 2 5 1 7 4]
 [1 7 4 9 3 6 8 2 5]
 [8 4 2 1 5 9 7 6 3]
 [3 6 9 7 1 2 4 5 8]
 [2 3 7 5 8 4 6 1 9]
 [7 5 8 4 9 1 2 3 6]
 [6 2 1 8 4 3 5 9 7]] 

[[3 7 2 4 9 6 5 1 8]
 [1 3 7 9 2 8 6 5 4]
 [5 8 9 3 7 2 4 6 1]
 [8 2 4 5 1 3 9 7 6]
 [6 9 8 7 4 1 3 2 5]
 [4 1 5 2 6 7 8 9 3]
 [2 4 1 6 8 5 7 3 9]
 [9 6 3 1 5 4 2 8 7]
 [7 5 6 8 3 9 1 4 2]] 

[[9 4 7 2 6 8 1 3 5]
 [6 7 4 8 9 3 2 5 1

In [10]:
latin_solver.est_log_total()

Array(63.663944, dtype=float32)

In [11]:
from jax import vmap, jit
from jax import numpy as jnp
import numpy as np

def solve(dims: int):
    latin_solver = LatinSquareSampler(n = dims, score_fn = Scorer.score_repetitive_columns)
    latin_solver.run_smc(no_samples = 100_000)
    return latin_solver.est_log_total() 

ans = [solve(dims) for dims in range(1, 12)]

Current loop value: 0.0
Current loop value: 0.0
Current loop value: 0.00046207010746002197
Current loop value: 0.0009228394483216107
Current loop value: 0.0014409529976546764
Current loop value: 0.0
Current loop value: 0.000373045913875103
Current loop value: 0.0007411172846332192
Current loop value: 0.0011003254912793636
Current loop value: 0.0014632996171712875
Current loop value: 0.0018735595513135195
Current loop value: 0.0
Current loop value: 0.000311901792883873
Current loop value: 0.0006266698474064469
Current loop value: 0.0009419576963409781
Current loop value: 0.0012581304181367159
Current loop value: 0.0015727865975350142
Current loop value: 0.0018906649202108383
Current loop value: 0.00223164027556777
Current loop value: 0.002657831646502018
Current loop value: 0.0
Current loop value: 0.0002643689513206482
Current loop value: 0.0005306614330038428
Current loop value: 0.000800865120254457
Current loop value: 0.001073599560186267
Current loop value: 0.0013490261044353247
Curr

In [12]:
for dims in range(1, 12):
    print(f"est log value = {ans[dims - 1]}, log_grouth_truth = {log_grouth_truth[dims - 1]}")

est log value = 0.0, log_grouth_truth = 0.0
est log value = 0.6931467056274414, log_grouth_truth = 0.6931471805599453
est log value = 2.4925355911254883, log_grouth_truth = 2.4849066497880004
est log value = 6.357021331787109, log_grouth_truth = 6.356107660695891
est log value = 12.006982803344727, log_grouth_truth = 11.989476363991853
est log value = 20.531368255615234, log_grouth_truth = 20.51524607852745
est log value = 31.61868667602539, log_grouth_truth = 31.74754634896515
est log value = 45.91276168823242, log_grouth_truth = 46.13891007716285
est log value = 62.660240173339844, log_grouth_truth = 63.87674183985093
est log value = 79.37001037597656, log_grouth_truth = 85.1927379585188
est log value = 92.71646118164062, log_grouth_truth = 59.61662636289668


#### Sudoku Solver

In [19]:
import jax
import jax.numpy as jnp
from jax import random

import numpy as np 

def prior_dist_logpdf(flat_board: jnp.ndarray):
    """
    Args:
        flat_board:
            (N*N,)      -> returns scalar
            (B, N*N)    -> returns (B,)
    """
    board_size = flat_board.shape[-1]
    n = 9

    board = flat_board.reshape(*flat_board.shape[:-1], n, n)

    # log_fact[k] = log(k!)
    log_fact = jnp.concatenate([
        jnp.array([0.0]),
        jnp.cumsum(jnp.log(jnp.arange(1, n + 1)))
    ])

    unfilled = jnp.sum(board == 0, axis=-1)

    log_prob_per_row = log_fact[unfilled]

    return -jnp.sum(log_prob_per_row, axis=-1)

def target_dist_logpdf(flat_board: jnp.array, score_fn, lam=4000):
    """
    Supports:
        (N,)  -> scalar
        (B,N) -> (B,)
    """
    if flat_board.ndim == 1:
        return -lam * score_fn(flat_board)
    else:
        scores = jax.vmap(score_fn)(flat_board)
        return -lam * scores

In [20]:
from jax import random
from jax import numpy as jnp
from functools import partial
import jax
import numpy as np 

class SudokuSolver(ISolver): 
    def __init__(self, configuration: jnp.array, score_fn):
        self.__n = 9
        self.configuration = configuration.reshape(9, 9)

        self.unfilled_indices = []
        self.unfilled_values = []

        all_values = set(range(1, 9 + 1))

        for row in self.configuration:
            row = row.tolist()

            filled = {x for x in row if x != 0}
            missing = sorted(all_values - filled)
            indices = [i for i, x in enumerate(row) if x == 0]

            self.unfilled_indices.append(jnp.array(indices))
            self.unfilled_values.append(jnp.array(missing))
        
        super().__init__(
            n = 9, 
            target_dist_logpdf = partial(target_dist_logpdf, score_fn = score_fn), 
            prior_dist_logpdf = prior_dist_logpdf, 
            proposed_fn = self.proposed_fn
        )

    def _generate_initial_state(self, key):
        board = self.configuration.copy()
    
        keys = random.split(key, len(self.unfilled_indices))
    
        for row, (idx, values, k) in enumerate(zip(self.unfilled_indices, self.unfilled_values, keys)):
            permuted = random.permutation(k, values)
    
            board = board.at[row, idx].set(permuted)
    
        return board.reshape(-1)

    def proposed_fn(self, flat_board, key):
        def proposed_one(flat_board, key): 
            board = flat_board.reshape(9, 9)
            row = np.random.randint(0, 9)
            key, sub_key = random.split(key, num = 2)
            
            if np.random.uniform() < 0.5:
                cols = jax.random.choice(sub_key, self.unfilled_indices[row], shape=(2,), replace=False)
                idx = self.unfilled_indices[row][cols]  
            else:
                idx = self.unfilled_indices[row]
            
            permuted = random.permutation(key, self.unfilled_values[row][idx])
            board = board.at[row, idx].set(permuted)
            return board.reshape(-1)

        if flat_board.ndim == 1:
            return proposed_one(flat_board, key)
        else:
            keys = random.split(key, flat_board.shape[0])
            return jax.vmap(proposed_one)(flat_board, keys)

Now let's try our solver with some puzzle!

In [21]:
from jax import numpy as jnp

problem = jnp.array([
    0,0,0,6,0,0,3,0,0,
    0,0,0,0,0,9,0,0,1,
    7,2,0,0,0,0,0,4,0,
    0,0,4,0,0,0,0,0,8,
    0,0,7,4,0,0,5,2,0,
    2,0,0,0,0,6,0,7,0,
    8,0,0,0,1,0,6,0,0,
    5,0,0,0,0,7,0,0,0,
    9,1,0,3,0,0,0,0,0,
])

sudoku_solver = SudokuSolver( 
    configuration = problem,
    score_fn = Scorer.score_sudoku
)

In [22]:
lam_list = sudoku_solver.run_smc(
    no_samples = 1_000_000
)

samples = sudoku_solver.get_current_sample_list()

Current loop value: 0.0
Current loop value: 2.9120594263076782e-05
Current loop value: 5.901051918044686e-05
Current loop value: 8.89367947820574e-05
Current loop value: 0.00011893201735801995
Current loop value: 0.0001492708979640156
Current loop value: 0.00018091435777023435
Current loop value: 0.00021580190514214337
Current loop value: 0.00025735230883583426
Current loop value: 0.00031126668909564614
Current loop value: 0.00038579030660912395
Current loop value: 0.0004927794216200709
Current loop value: 0.0006522876210510731


In [23]:
sudoku_solver.est_log_total()

Array(-104018.21, dtype=float32)

In [24]:
from tqdm import tqdm
for sample in tqdm(samples[:10]):
    print(Scorer.score_sudoku(sample))

 30%|██████████████▍                                 | 3/10 [00:00<00:00, 12.20it/s]

26
26
26


 50%|████████████████████████                        | 5/10 [00:00<00:00, 13.22it/s]

26
26
26


 90%|███████████████████████████████████████████▏    | 9/10 [00:00<00:00, 14.00it/s]

26
26
25


100%|███████████████████████████████████████████████| 10/10 [00:00<00:00, 13.47it/s]

26


In [26]:
samples[0].reshape(9,9)

Array([[8, 5, 1, 6, 2, 4, 3, 9, 7],
       [3, 6, 4, 7, 8, 9, 5, 2, 1],
       [7, 2, 5, 1, 6, 9, 8, 4, 3],
       [1, 9, 4, 5, 7, 2, 6, 3, 8],
       [8, 6, 7, 4, 3, 1, 5, 2, 9],
       [2, 3, 4, 8, 9, 6, 1, 7, 5],
       [8, 7, 4, 2, 1, 3, 6, 5, 9],
       [5, 3, 2, 9, 6, 7, 9, 9, 4],
       [9, 1, 6, 3, 4, 8, 2, 7, 5]], dtype=int32)